<a href="https://colab.research.google.com/github/sofiascarvalho/Projeto_Integrador_AWS/blob/main/Analise_Sentimento_AWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **0. Instalação de Pacotes**

Esta seção contém todas as instalações de bibliotecas necessárias para o notebook. Execute esta célula para garantir que todos os pacotes estejam disponíveis.

In [ ]:
# Instalações de bibliotecas
!pip install vaderSentiment
!pip install leia-br
!pip install transformers torch

## **0.1 Importação de Bibliotecas Essenciais e Inicializações Globais**

Nesta seção, importamos todas as bibliotecas Python que serão utilizadas e inicializamos objetos globais como os analisadores de sentimento para VADER, LeIA e Transformers.

In [ ]:
# Importação das bibliotecas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import pandas as pd
import unicodedata
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from LeIA import SentimentIntensityAnalyzer as SentimentIntensityAnalyzer_leia
from transformers import pipeline
from tqdm.auto import tqdm

# Inicialização dos analisadores de sentimento
sid = SentimentIntensityAnalyzer()
sid_leia = SentimentIntensityAnalyzer_leia()
# Atenção: ao usar o pipeline de transformers, se você ver um aviso sobre 'HF_TOKEN',
# considere adicionar seu token da Hugging Face aos segredos do Colab para evitar limitações de taxa.
analyzer = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

## **0.2 Funções Auxiliares**

Esta seção agrupa todas as funções auxiliares definidas no notebook, como funções de análise de DataFrame, manipulação de texto, classificação de sentimento e algoritmos de ordenação (Selection Sort).

In [ ]:
def analise_df(nome, df):
  print(f"\n{'='*50}")
  print(f"\nTABELA: {nome}")
  print(f"\n{'='*50}")

  print("\nHEAD")
  display(df.head())

  print("\nTAIL")
  display(df.tail())

  print("\nINFO")
  df.info()

  print("\nDESCRIBE")
  display(df.describe())

  print("\nItens Nulos")
  print(df.isnull())

def combine_review_text(row):
    title = row['review_comment_title']
    message = row['review_comment_message']

    # Convert to string and handle NaN as empty string
    title_str = str(title) if pd.notna(title) else ''
    message_str = str(message) if pd.notna(message) else ''

    if title_str and message_str:
        return f"{title_str} - {message_str}"
    elif title_str:
        return title_str
    elif message_str:
        return message_str
    else:
        return ''

def strip_accents(text):
   """Remove accents from a string."""
   # Ensure text is a string before processing
   if isinstance(text, str):
       text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
   return text


def classificacao (pontuacao):
    """Classifies a sentiment score into textual categories."""
    if pontuacao < -0.6:
        return "muito negativo"
    elif pontuacao >= -0.6 and pontuacao < -0.2:
        return "negativo"
    elif pontuacao >= -0.2 and pontuacao < 0.2:
        return "neutro"
    elif pontuacao >= 0.2 and pontuacao < 0.6:
        return "positivo"
    else:
        return "muito positivo"

def classificacao_score (pontuacao):
    """Maps a sentiment score to a 1-5 star rating."""
    if pontuacao < -0.6:
        return 1
    elif pontuacao >= -0.6 and pontuacao < -0.2:
        return 2
    elif pontuacao >= -0.2 and pontuacao < 0.2:
        return 3
    elif pontuacao >= 0.2 and pontuacao < 0.6:
        return 4
    else:
        return 5

def calcular_sentimento_vader(texto):
    # O texto combinado já lida com NaN e strings vazias, mas a validação interna é boa.
    if not isinstance(texto, str) or texto.strip() == '':
        return 0.0
    return sid.polarity_scores(texto)['compound']

def sentimento_score_por_review_score(score):
    """Calcula uma pontuação numérica de sentimento (-1 a 1) a partir do review_score (1 a 5)."""
    # Mapeamento do review_score para uma escala de -1 a 1
    if score >= 4:
        return round((score - 3) / 2, 2) # Ex: 4 -> 0.5, 5 -> 1.0
    elif score == 3:
        return 0.0 # Neutro
    else:
        return round((score - 3) / 2, 2) # Ex: 2 -> -0.5, 1 -> -1.0

def calcular_sentimento_leia(row):
    texto = str(row['review_text_combined']).strip() # Usar review_text_combined

    if not texto:
        # Se não houver texto, use o review_score para inferir o sentimento
        return sentimento_score_por_review_score(int(row['review_score']))

    # Se houver texto, use o LeIA normalmente
    scores = sid_leia.polarity_scores(texto)
    return scores['compound']

def map_transformer_label_to_score(label_str):
    """Maps transformer string labels (e.g., '3 stars') to a numerical sentiment score (-1 to 1)."""
    if '1 star' in label_str:
        return -1.0
    elif '2 stars' in label_str:
        return -0.5
    elif '3 stars' in label_str:
        return 0.0
    elif '4 stars' in label_str:
        return 0.5
    elif '5 stars' in label_str:
        return 1.0
    return 0.0 # Default or error case, assuming neutral

def selection_sort_by_length(data):
    n = len(data)
    # Criamos uma cópia da lista original para não modificar o DataFrame diretamente
    # e para trabalhar com os textos reais para ordenação.
    sorted_data = list(data)

    for i in range(n):
        # Encontra o índice do maior elemento restante
        max_idx = i
        for j in range(i + 1, n):
            if len(sorted_data[j]) > len(sorted_data[max_idx]): # Alterado para > para ordenar do maior para o menor
                max_idx = j

        # Troca o elemento encontrado com o primeiro elemento não classificado
        sorted_data[i], sorted_data[max_idx] = sorted_data[max_idx], sorted_data[i]
    return sorted_data

def selection_sort_by_value_descending(data_with_values):
    """Sorts a list of (value, text) tuples in descending order by value using Selection Sort."""
    n = len(data_with_values)
    sorted_data = list(data_with_values)

    for i in range(n):
        # Encontra o índice do maior valor restante
        max_idx = i
        for j in range(i + 1, n):
            if sorted_data[j][0] > sorted_data[max_idx][0]: # Compara os valores (primeiro elemento da tupla)
                max_idx = j

        # Troca o elemento encontrado com o primeiro elemento não classificado
        sorted_data[i], sorted_data[max_idx] = sorted_data[max_idx], sorted_data[i]
    return sorted_data

#**ANALISE DESCRITIVA** - Banco de Dados de avaliações *AWS*

## **2. Análise Descritiva**

# **1. Carregamento e Entendimento dos Dados**
Nessa etapa realizaremos o carregamento organizado de todas as tabelas do projeto no Python (Google Colab ou VS Code), além de compreender o que cada tabela representa, como se relacionam e qual o papel de cada uma no fluxo de um e-commerce. Também construiremos um dicionário de dados simples, descrevendo cada variável, seu tipo, significado e função no conjunto de informações.

### **Montagem do Google Drive**
Esta célula executa o comando para montar o Google Drive no ambiente do Colab, permitindo o acesso aos arquivos armazenados na nuvem, que contêm os datasets do projeto.

## **1.2 Carregamento dos dados**
Nessa etapa carregaremos todas as tabelas do projeto no Python utilizando a biblioteca Pandas. Utilize
caminhos corretos e específicos de suas pastas. Carregue as nove tabelas: pedidos, itens do pedido,
clientes, produtos, pagamentos, avaliações, vendedores, geolocalização e auxiliar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **Carregamento dos Datasets Iniciais**
Nesta célula, todas as tabelas CSV são lidas e carregadas em DataFrames do Pandas. Cada DataFrame recebe um nome (`df_avaliacoes`, `df_clientes`, etc.) para facilitar a referência e manipulação posterior no código.

In [ ]:
#avaliações
df_avaliacoes = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/avaliacoes.csv')

#clientes
df_clientes = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/clientes.csv')

#geolocalizacao
df_geolocalizacao = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/geolocalizacao.csv')

#itens_pedidos
df_itens_pedidos = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/itens_pedidos.csv')

#pagamentos
df_pagamentos = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/pagamentos.csv')

#pedidos
df_pedidos = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/pedidos.csv')

#produtos
df_produtos = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/produtos.csv')

#tabela_auxiliar
df_tabela_auxiliar = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/tabela_auxiliar.csv')

#vendedores
df_vendedores = pd.read_csv('/content/drive/MyDrive/Banco_de_Dados_PII1_AWS/Banco_de_Dados_PII3_AWS/vendedores.csv')

### **Criação do Dicionário de DataFrames**
Esta célula organiza todos os DataFrames carregados em um dicionário chamado `tabelas`. Isso permite uma iteração e manipulação mais eficiente de todos os conjuntos de dados de forma programática.

In [ ]:
tabelas = {
    'avaliacoes': df_avaliacoes,
    'clientes': df_clientes,
    'geolocalizacao': df_geolocalizacao,
    'itens_pedidos': df_itens_pedidos,
    'pagamentos': df_pagamentos,
    'pedidos': df_pedidos,
    'produtos': df_produtos,
    'tabela_auxiliar': df_tabela_auxiliar,
    'vendedores': df_vendedores
}

### **Execução da Análise Exploratória para Todas as Tabelas**
Esta célula itera sobre o dicionário `tabelas` e aplica a função `analise_df` a cada DataFrame. Isso gera uma saída detalhada para cada uma das nove tabelas, fornecendo uma visão abrangente de sua estrutura, conteúdo e integridade.

In [ ]:
for nome, df in tabelas.items():
    analise_df(nome, df)

In [ ]:
df_final = df_clientes.merge(
    df_pedidos, on='customer_id',
).merge(
    df_itens_pedidos, on='order_id',
).merge(
    df_produtos, on='product_id',
).merge(
    df_avaliacoes, on='order_id',
).merge(
    df_pagamentos, on='order_id',
).merge(
    df_vendedores, on='seller_id',
)

### **Visão Geral do DataFrame Final**
Esta célula exibe as dimensões (linhas e colunas) e as primeiras linhas do DataFrame `df_final`, que foi criado pela junção de várias tabelas. Isso nos permite ter uma rápida noção da estrutura dos dados mesclados.

In [ ]:
print(df_final.shape)
df_final.head()

### **Resumo Básico das Tabelas**
Esta célula calcula e exibe um resumo básico para cada DataFrame original, incluindo o número de linhas, colunas, o total de valores nulos e o número de linhas duplicadas. É uma visão rápida da integridade estrutural de cada tabela.

In [ ]:
resumo = []
for nome, df in tabelas.items():
    resumo.append({
        "Tabela":      nome,
        "Linhas":      df.shape[0],
        "Colunas":     df.shape[1],
        "Nulos Total": df.isnull().sum().sum(),
        "Duplicatas":  df.duplicated().sum(),
    })

df_resumo = pd.DataFrame(resumo)
display(df_resumo)

### **Resumo Detalhado das Tabelas (incluindo Duplicatas por ID)**
Esta célula aprimora o resumo das tabelas adicionando a porcentagem de nulos e, crucialmente, identificando duplicatas com base nas colunas que provavelmente são IDs. Isso ajuda a detectar problemas de integridade de dados mais específicos em chaves primárias.

In [ ]:


resumo = []

for nome, df in tabelas.items():
    # 1. Identifica colunas que provavelmente são IDs (contêm 'id' ou 'customer', 'order', etc)
    # Isso ajuda a achar duplicatas "escondidas" por IDs únicos
    cols_id = [c for c in df.columns if 'id' in c.lower() or 'pk' in c.lower()]

    # 2. Calcula duplicatas na chave principal (se existir)
    # Se houver mais de um ID, pegamos o primeiro (geralmente o ID da própria tabela)
    dup_chave = 0
    if cols_id:
        dup_chave = df.duplicated(subset=[cols_id[0]]).sum()

    resumo.append({
        "Tabela":       nome,
        "Linhas":       df.shape[0],
        "Colunas":      df.shape[1],
        "Nulos Total":  df.isnull().sum().sum(),
        "Nulos %":      f"{(df.isnull().sum().sum() / (df.size) * 100):.1f}%",
        "Dup. Linha":   df.duplicated().sum(),
        "Dup. ID":      dup_chave, # <--- O "pulo do gato" está aqui
        "Chave Usada":  cols_id[0] if cols_id else "N/A"
    })

df_resumo = pd.DataFrame(resumo)
display(df_resumo)


### **Conversão de Colunas de Data em `df_avaliacoes`**
Esta célula converte as colunas `review_creation_date` e `review_answer_timestamp` do DataFrame `df_avaliacoes` para o tipo de dado `datetime`. Isso é essencial para análises temporais, como duração entre eventos e tendências ao longo do tempo.

In [ ]:
df_avaliacoes['review_creation_date']    = pd.to_datetime(df_avaliacoes['review_creation_date'])
df_avaliacoes['review_answer_timestamp'] = pd.to_datetime(df_avaliacoes['review_answer_timestamp'])

print('Info of df_avaliacoes after datetime conversion:')
df_avaliacoes.info()
print('\nHead of df_avaliacoes after datetime conversion:')
display(df_avaliacoes.head())

### **Tratamento e Combinação de Texto em Avaliações**
Esta célula agora realiza o seguinte:

1.  **Combinação de Título e Mensagem:** Cria uma nova coluna `review_text_combined` que concatena `review_comment_title` e `review_comment_message`. Se apenas um estiver presente, ele é usado. Se ambos forem nulos, a coluna combinada fica vazia.
2.  **Filtragem de Avaliações Sem Texto:** Remove as linhas do DataFrame `df_avaliacoes` onde a nova coluna `review_text_combined` está vazia, ou seja, avaliações que não possuíam título nem mensagem original.
3.  **Resumo da Contagem:** Exibe a quantidade total de avaliações originais, a quantidade de avaliações que restaram após a filtragem (com algum texto combinado) e a quantidade de avaliações que foram removidas por não terem conteúdo textual.

In [ ]:
# A conversão para datetime já foi feita na célula XBjzPiiIjXFc. Removendo duplicação.
# df_avaliacoes['review_creation_date']    = pd.to_datetime(df_avaliacoes['review_creation_date'])
# df_avaliacoes['review_answer_timestamp'] = pd.to_datetime(df_avaliacoes['review_answer_timestamp'])

original_total_reviews = len(df_avaliacoes)



df_avaliacoes['review_text_combined'] = df_avaliacoes.apply(combine_review_text, axis=1)

# Filter df_avaliacoes to remove rows where 'review_text_combined' is empty
df_avaliacoes = df_avaliacoes[df_avaliacoes['review_text_combined'] != ''].copy()

# df_avaliacoes_com_texto now includes all rows that have any combined text
df_avaliacoes_com_texto = df_avaliacoes.copy()

print(f"Total de avaliações originais: {original_total_reviews:,}")
print(f"Avaliações com texto combinado (título ou mensagem): {len(df_avaliacoes):,}")
print(f"Avaliações removidas (sem título e sem comentário original): {original_total_reviews - len(df_avaliacoes):,}")

### **Padronização de Texto em Avaliações (Minúsculas e Remoção de Acentos)**
Esta célula padroniza o texto na coluna `review_text_combined` convertendo todo o conteúdo para minúsculas e removendo caracteres acentuados. Isso é crucial para análises de texto, garantindo que palavras como 'Ótimo' e 'otimo' sejam tratadas como a mesma entidade.

In [ ]:
# Convert to lowercase and remove accents from the combined text column
# The 'review_text_combined' column should already contain only strings or empty strings from the previous step
df_avaliacoes['review_text_combined'] = df_avaliacoes['review_text_combined'].str.lower().apply(strip_accents)

print("Coluna 'review_text_combined' padronizada (minúsculas e sem acentos).")
print('\nExemplo de df_avaliacoes.head() após padronização:')
display(df_avaliacoes[['review_text_combined', 'review_comment_title', 'review_comment_message']].head())

### **Conversão e Análise de Colunas de Data em `df_pedidos`**
Esta célula converte múltiplas colunas de data do DataFrame `df_pedidos` para o tipo `datetime`. Em seguida, ela verifica a contagem de pedidos com datas de entrega nulas (`order_delivered_customer_date`) agrupadas pelo status do pedido, confirmando que esses nulos estão associados a pedidos ainda não entregues ou cancelados, e não a erros de dados.

In [ ]:
colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in colunas_data:
    df_pedidos[col] = pd.to_datetime(df_pedidos[col])

# Confirma que nulos são pedidos não entregues (não é erro)
print("Nulos em 'data de entrega' por status:")
print(df_pedidos[df_pedidos['order_delivered_customer_date'].isnull()]['order_status'].value_counts())
print(f"\nTotal de pedidos mantidos: {len(df_pedidos):,}")

### **Tratamento de Categorias Nulas em `df_produtos`**
Esta célula preenche os valores nulos na coluna `product_category_name` do DataFrame `df_produtos` com a string 'sem_categoria'. Isso garante que todos os produtos tenham uma categoria, evitando problemas em análises ou visualizações que dependem dessa informação.

In [ ]:
nulos_categoria = df_produtos['product_category_name'].isnull().sum()

df_produtos['product_category_name'] = df_produtos['product_category_name'].fillna('sem_categoria')

print(f"Categorias preenchidas com 'sem_categoria': {nulos_categoria}")
print(f"Total de produtos: {len(df_produtos):,}")

### **Verificação Final de Nulos e Duplicatas em Tabelas Selecionadas**
Esta célula executa uma verificação rápida de valores nulos e linhas duplicadas nas tabelas `itens_pedidos`, `pagamentos`, `clientes`, `vendedores` e `tabela_auxiliar`. O objetivo é confirmar se, após os tratamentos anteriores e a análise inicial, essas tabelas estão limpas e prontas para uso.

In [ ]:
for nome, df in {
    "itens_pedidos":   df_itens_pedidos,
    "pagamentos":      df_pagamentos,
    "clientes":        df_clientes,
    "vendedores":      df_vendedores,
    "tabela_auxiliar": df_tabela_auxiliar
}.items():
    nulos = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    print(f"{nome:20s} → nulos: {nulos} | duplicatas: {dups}")

## **3. Análise de Sentimento**

Nesta seção, exploraremos os resultados da análise de sentimento utilizando três modelos distintos: VADER, LeIA e Transformers. Avaliaremos suas pontuações, classificações e como se comparam com as avaliações originais dos clientes.

### **Visualização dos Resultados da Análise de Sentimento (VADER)**
Esta célula exibe as colunas `review_text_combined`, `review_score` e a recém-criada `pontuacao_sentimento_vader` do DataFrame `df_avaliacoes`. Isso permite uma inspeção rápida dos comentários dos clientes e seus respectivos scores de sentimento calculados.

In [ ]:
df_avaliacoes['pontuacao_sentimento_vader'] = df_avaliacoes['review_text_combined'].apply(calcular_sentimento_vader)

print("Coluna 'pontuacao_sentimento_vader' adicionada ao DataFrame df_avaliacoes.")
display(df_avaliacoes[['review_text_combined', 'review_score', 'pontuacao_sentimento_vader']].head())

In [ ]:
df_avaliacoes['pontuacao_sentimento_leia'] = (
   df_avaliacoes.apply(calcular_sentimento_leia, axis=1)
)

print("Coluna 'pontuacao_sentimento_leia' adicionada ao DataFrame df_avaliacoes.")
display(df_avaliacoes[['review_text_combined', 'review_score', 'pontuacao_sentimento_leia']].head())

## Transformers

### **Cálculo da Pontuação de Sentimento com `transformers`**
Esta célula define uma função para converter as saídas de 'estrelas' do modelo `transformers` em uma pontuação numérica contínua de sentimento (-1 a 1), facilitando a comparação com VADER e LeIA. Em seguida, aplica essa função à coluna `review_comment_message`.

In [ ]:
review_texts_list = df_avaliacoes['review_text_combined'].tolist()

all_sentiment_scores = []
batch_size = 256 # Ajuste conforme a memória disponível e o tamanho do dataset

r, resultados_pipeline = [], []
for i in tqdm(range(0, len(review_texts_list), batch_size), desc="Analisando sentimentos em batches"):
    batch = review_texts_list[i:i + batch_size]
    resultados_pipeline.extend(analyzer(batch, truncation=True))


for result in tqdm(resultados_pipeline, total=len(review_texts_list), desc="Processando resultados do pipeline"):
    score = result['label']
    all_sentiment_scores.append(score)

# Atribui os resultados ao DataFrame principal
df_avaliacoes['pontuacao_sentimento_transformers'] = all_sentiment_scores

print("Coluna 'pontuacao_sentimento_transformers' adicionada ao DataFrame com sucesso.")
print("Exemplo de df_avaliacoes após análise com transformers:")
display(df_avaliacoes[['review_text_combined', 'review_score', 'pontuacao_sentimento_transformers']].head())

### **Classificação e Geração de Scores de 1 a 5 para Todos os Modelos de Sentimento**

Esta célula aplica as funções de classificação (`classificacao` e `classificacao_score`) a todas as pontuações de sentimento (`vader`, `leia`, `transformers`), criando novas colunas para a categoria textual (`classe_`) e o score de 1 a 5 (`score_classificado_`) para cada modelo. Isso permite uma padronização e comparação mais fácil dos resultados.

In [ ]:
# Aplicar as funções de classificação aos scores de sentimento
df_avaliacoes['classe_vader'] = df_avaliacoes['pontuacao_sentimento_vader'].apply(classificacao)
df_avaliacoes['score_classificado_vader'] = df_avaliacoes['pontuacao_sentimento_vader'].apply(classificacao_score)

df_avaliacoes['classe_leia_nova'] = df_avaliacoes['pontuacao_sentimento_leia'].apply(classificacao)
df_avaliacoes['score_classificado_leia'] = df_avaliacoes['pontuacao_sentimento_leia'].apply(classificacao_score)

# Verifica se a coluna ainda contém strings antes de converter, para evitar o TypeError em execuções repetidas.
if df_avaliacoes['pontuacao_sentimento_transformers'].dtype == 'object':
    # Converta as labels de string do transformers para scores numéricos
    df_avaliacoes['pontuacao_sentimento_transformers'] = df_avaliacoes['pontuacao_sentimento_transformers'].apply(map_transformer_label_to_score)

df_avaliacoes['classe_transformers'] = df_avaliacoes['pontuacao_sentimento_transformers'].apply(classificacao)
df_avaliacoes['score_classificado_transformers'] = df_avaliacoes['pontuacao_sentimento_transformers'].apply(classificacao_score)

print("Novas colunas de classificação e score adicionadas para Vader, LeIA e Transformers.")

print("\n--- Exemplo de Classificação Vader ---")
display(df_avaliacoes[['review_text_combined', 'pontuacao_sentimento_vader', 'classe_vader', 'score_classificado_vader']].head())

print("\n--- Exemplo de Classificação LeIA ---")
display(df_avaliacoes[['review_text_combined', 'pontuacao_sentimento_leia', 'classe_leia_nova', 'score_classificado_leia']].head())

print("\n--- Exemplo de Classificação Transformers ---")
display(df_avaliacoes[['review_text_combined', 'pontuacao_sentimento_transformers', 'classe_transformers', 'score_classificado_transformers']].head())


## Guia de Interpretação dos Gráficos de Análise de Sentimento

Para avaliar a performance e a coerência de cada modelo de análise de sentimento (VADER, LeIA, Transformers), utilizaremos três tipos principais de gráficos. Veja como interpretá-los:

### 1. Histograma da Pontuação de Sentimento Contínua (`pontuacao_sentimento_`)

*   **O que mostra:** A distribuição das pontuações de sentimento em uma escala contínua, geralmente de -1.0 (muito negativo) a 1.0 (muito positivo). O eixo X representa a pontuação e o eixo Y, a frequência (quantas avaliações caíram em cada faixa de pontuação).
*   **O que procurar:**
    *   **Picos:** Onde a maioria das avaliações está concentrada (ex: muitos neutros, muitos positivos).
    *   **Assimetria (Skewness):** Se a distribuição é mais inclinada para o lado positivo ou negativo.
    *   **Amplitude:** A variação das pontuações. Um modelo mais sensível pode ter uma distribuição mais ampla.

### 2. Gráfico de Barras de Classificação em Estrelas (`score_classificado_`)

*   **O que mostra:** A contagem de avaliações que foram classificadas em cada uma das 5 estrelas pelo modelo de sentimento. O eixo X representa a classificação em estrelas (1 a 5) e o eixo Y, o número de avaliações.
*   **O que procurar:**
    *   **Proporções:** Quais categorias de estrelas são mais ou menos frequentes de acordo com a classificação do modelo.
    *   **Comparação:** Se a distribuição de estrelas classificadas faz sentido em relação ao que você esperaria dos dados.

### 3. Mapa de Calor (Heatmap) de Comparação (`review_score` vs. `score_classificado_`)

*   **O que mostra:** Uma matriz de confusão visual que compara a `review_score` original (a avaliação em estrelas dada pelo cliente) com a `score_classificado_` do modelo (a classificação em estrelas atribuída pelo algoritmo de sentimento).
    *   **Eixo Y:** `review_score` original.
    *   **Eixo X:** `score_classificado_` do modelo.
    *   **Cores e Números:** A intensidade da cor e os números dentro de cada célula indicam quantas avaliações caíram nessa combinação específica (ex: quantas avaliações de 5 estrelas originais foram classificadas como 4 estrelas pelo modelo).
*   **O que procurar:**
    *   **Diagonal Principal:** As células na diagonal (superior esquerda para inferior direita) representam os casos em que o modelo concordou com a `review_score` original. Quanto maiores os números e mais fortes as cores nessa diagonal, melhor o alinhamento do modelo.
    *   **Fora da Diagonal:** As células fora da diagonal indicam discrepâncias. Analisá-las ajuda a entender onde o modelo 'erra' (ex: um modelo classifica como 1 estrela o que o cliente deu 5, ou vice-versa). Isso é útil para identificar vieses ou áreas onde o modelo pode estar lutando para interpretar o sentimento corretamente.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_avaliacoes['pontuacao_sentimento_vader'], bins=20, kde=True)
plt.title('Distribuição da Pontuação de Sentimento do VADER')
plt.xlabel('Pontuação de Sentimento (-1 a 1)')
plt.ylabel('Frequência')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribuição das `score_classificado_vader`

Agora, vamos ver a distribuição das classificações em estrelas (1 a 5) geradas pelo modelo VADER.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='score_classificado_vader', data=df_avaliacoes, palette='viridis', order=sorted(df_avaliacoes['score_classificado_vader'].unique()))
plt.title('Contagem de Classificações em Estrelas do VADER')
plt.xlabel('Classificação (1 a 5 Estrelas)')
plt.ylabel('Número de Avaliações')
plt.show()

### Comparação entre `score_classificado_vader` e `review_score`

Para verificar a coerência do VADER, é fundamental comparar a classificação do modelo com a `review_score` original.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(pd.crosstab(df_avaliacoes['review_score'], df_avaliacoes['score_classificado_vader']), cmap='Blues', annot=True, fmt='d')
plt.title('Comparação: review_score (Original) vs. score_classificado_vader')
plt.xlabel('Classificação VADER (1-5 Estrelas)')
plt.ylabel('Review Score Original (1-5 Estrelas)')
plt.show()

# Exemplo de reviews onde o score do VADER difere do score original
discrepancias_vader = df_avaliacoes[df_avaliacoes['review_score'] != df_avaliacoes['score_classificado_vader']]
print("\nAlguns exemplos de avaliações onde a classificação do VADER difere do score original:")
display(discrepancias_vader[['review_text_combined', 'review_score', 'pontuacao_sentimento_vader', 'score_classificado_vader']].sample(5))

## Análise e Visualização dos Resultados do Sentimento LeIA

Agora, vamos visualizar as pontuações e classificações geradas pelo modelo LeIA para entender melhor a distribuição do sentimento e compará-las com as `review_score`s originais.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_avaliacoes['pontuacao_sentimento_leia'], bins=20, kde=True)
plt.title('Distribuição da Pontuação de Sentimento do LeIA')
plt.xlabel('Pontuação de Sentimento (-1 a 1)')
plt.ylabel('Frequência')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribuição das `score_classificado_leia`

Agora, vamos ver a distribuição das classificações em estrelas (1 a 5) geradas pelo modelo LeIA.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='score_classificado_leia', data=df_avaliacoes, palette='viridis', order=sorted(df_avaliacoes['score_classificado_leia'].unique()))
plt.title('Contagem de Classificações em Estrelas do LeIA')
plt.xlabel('Classificação (1 a 5 Estrelas)')
plt.ylabel('Número de Avaliações')
plt.show()

### Comparação entre `score_classificado_leia` e `review_score`

Para verificar a coerência do LeIA, é fundamental comparar a classificação do modelo com a `review_score` original.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(pd.crosstab(df_avaliacoes['review_score'], df_avaliacoes['score_classificado_leia']), cmap='Blues', annot=True, fmt='d')
plt.title('Comparação: review_score (Original) vs. score_classificado_leia')
plt.xlabel('Classificação LeIA (1-5 Estrelas)')
plt.ylabel('Review Score Original (1-5 Estrelas)')
plt.show()

# Exemplo de reviews onde o score do LeIA difere do score original
discrepancias_leia = df_avaliacoes[df_avaliacoes['review_score'] != df_avaliacoes['score_classificado_leia']]
print("\nAlguns exemplos de avaliações onde a classificação do LeIA difere do score original:")
display(discrepancias_leia[['review_text_combined', 'review_score', 'pontuacao_sentimento_leia', 'score_classificado_leia']].sample(5))

## Análise e Visualização dos Resultados do Sentiment Transformers

Vamos agora visualizar as pontuações e classificações geradas pelo modelo `transformers` para entender melhor a distribuição do sentimento e compará-las com as `review_score`s originais.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_avaliacoes['pontuacao_sentimento_transformers'], bins=20, kde=True)
plt.title('Distribuição da Pontuação de Sentimento do Transformers')
plt.xlabel('Pontuação de Sentimento (-1 a 1)')
plt.ylabel('Frequência')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribuição das `score_classificado_transformers`

Agora, vamos ver a distribuição das classificações em estrelas (1 a 5) geradas pelo modelo `transformers`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='score_classificado_transformers', data=df_avaliacoes, palette='viridis', order=sorted(df_avaliacoes['score_classificado_transformers'].unique()))
plt.title('Contagem de Classificações em Estrelas do Transformers')
plt.xlabel('Classificação (1 a 5 Estrelas)')
plt.ylabel('Número de Avaliações')
plt.show()

### Comparação entre `score_classificado_transformers` e `review_score`

Para verificar a coerência, é fundamental comparar a classificação do modelo `transformers` com a `review_score` original (a avaliação do cliente em estrelas). Uma matriz de confusão ou um gráfico de dispersão pode nos ajudar a visualizar isso.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(pd.crosstab(df_avaliacoes['review_score'], df_avaliacoes['score_classificado_transformers']), cmap='Blues', annot=True, fmt='d')
plt.title('Comparação: review_score (Original) vs. score_classificado_transformers')
plt.xlabel('Classificação Transformers (1-5 Estrelas)')
plt.ylabel('Review Score Original (1-5 Estrelas)')
plt.show()

# Exemplo de reviews onde o score do transformers difere do score original
discrepancias = df_avaliacoes[df_avaliacoes['review_score'] != df_avaliacoes['score_classificado_transformers']]
print("\nAlguns exemplos de avaliações onde a classificação do Transformers difere do score original:")
display(discrepancias[['review_text_combined', 'review_score', 'pontuacao_sentimento_transformers', 'score_classificado_transformers']].sample(5))

## **4. Comparação Direta dos Modelos de Análise de Sentimento**

Nesta seção, realizaremos uma comparação visual direta entre os três modelos de análise de sentimento (VADER, LeIA e Transformers) para entender suas semelhanças e diferenças.

### **Distribuição Comparativa das Classificações de Sentimento (1-5 Estrelas)**

Este gráfico de barras agrupa as classificações de sentimento de 1 a 5 estrelas geradas por cada modelo, permitindo visualizar como cada um distribui o sentimento geral. Isso ajuda a identificar se algum modelo tende a ser mais otimista, pessimista ou neutro em comparação com os outros.

In [ ]:
df_comparativo_classificacao = df_avaliacoes[[
    'score_classificado_vader',
    'score_classificado_leia',
    'score_classificado_transformers'
]].melt(var_name='Modelo', value_name='Classificação em Estrelas')

plt.figure(figsize=(12, 7))
sns.countplot(data=df_comparativo_classificacao, x='Classificação em Estrelas', hue='Modelo', palette='viridis')
plt.title('Distribuição Comparativa das Classificações de Sentimento (1-5 Estrelas)')
plt.xlabel('Classificação em Estrelas')
plt.ylabel('Número de Avaliações')
plt.legend(title='Modelo')
plt.grid(axis='y', alpha=0.75)
plt.show()

### **Ordenação de Avaliações por Tamanho (Selection Sort)**

Esta célula aplica o algoritmo Selection Sort para ordenar as avaliações com base no comprimento do texto (do maior para o menor). Isso permite identificar rapidamente os comentários mais detalhados e extensos, que podem conter informações mais ricas.

In [ ]:
# Aplicar o selection sort na coluna 'review_text_combined' e mostrar os primeiros 10 resultados
# Fazemos uma cópia para evitar warnings do pandas sobre SettingWithCopyWarning
reviews_to_sort = df_avaliacoes['review_text_combined'].copy()

sorted_reviews = selection_sort_by_length(reviews_to_sort.tolist())

print("Comentários ordenados pelo tamanho (maior para o menor):") # Mensagem de impressão atualizada
for i, review in enumerate(sorted_reviews[:20]): # Mostrar os primeiros 20 para ter uma ideia
    print(f"({len(review)} caracteres) {review}")

### **Ordenação por Pontuação de Sentimento com Selection Sort**

Agora, vamos aplicar o `selection_sort` para ordenar as avaliações com base na `pontuacao_sentimento_transformers`. Isso nos permitirá ver os comentários mais positivos e mais negativos de acordo com a análise do modelo.

In [ ]:
# Function selection_sort_by_value_descending (Moved to consolidated functions block)

# Preparar os dados: lista de tuplas (score, review_text_combined)
data_for_sorting = list(zip(df_avaliacoes['pontuacao_sentimento_transformers'], df_avaliacoes['review_text_combined']))

# Aplicar o selection sort
sorted_sentiment_reviews = selection_sort_by_value_descending(data_for_sorting)

print("Comentários ordenados pela pontuação de sentimento (mais positiva para mais negativa - Transformers):")
for score, review in sorted_sentiment_reviews[:10]: # Mostrar os 10 mais positivos
    print(f"({score:.2f}) {review}")

print("\n" + "="*70 + "\n")

print("Comentários ordenados pela pontuação de sentimento (mais negativa para mais positiva - Transformers):")
for score, review in sorted_sentiment_reviews[-10:]: # Mostrar os 10 mais negativos
    print(f"({score:.2f}) {review}")

### **Ordenação por Pontuação de Sentimento com Selection Sort (VADER)**

Agora, vamos aplicar o `selection_sort` para ordenar as avaliações com base na `pontuacao_sentimento_vader`. Isso nos permitirá ver os comentários mais positivos e mais negativos de acordo com a análise do modelo VADER.

In [ ]:
# Preparar os dados: lista de tuplas (score, review_text_combined)
data_for_sorting_vader = list(zip(df_avaliacoes['pontuacao_sentimento_vader'], df_avaliacoes['review_text_combined']))

# Aplicar o selection sort
sorted_sentiment_reviews_vader = selection_sort_by_value_descending(data_for_sorting_vader)

print("Comentários ordenados pela pontuação de sentimento (mais positiva para mais negativa - VADER):")
for score, review in sorted_sentiment_reviews_vader[:10]: # Mostrar os 10 mais positivos
    print(f"({score:.2f}) {review}")

print("\n" + "="*70 + "\n")

print("Comentários ordenados pela pontuação de sentimento (mais negativa para mais positiva - VADER):")
for score, review in sorted_sentiment_reviews_vader[-10:]: # Mostrar os 10 mais negativos
    print(f"({score:.2f}) {review}")

### **Ordenação por Pontuação de Sentimento com Selection Sort (LeIA)**

Agora, vamos aplicar o `selection_sort` para ordenar as avaliações com base na `pontuacao_sentimento_leia`. Isso nos permitirá ver os comentários mais positivos e mais negativos de acordo com a análise do modelo LeIA.

In [ ]:
# Preparar os dados: lista de tuplas (score, review_text_combined)
data_for_sorting_leia = list(zip(df_avaliacoes['pontuacao_sentimento_leia'], df_avaliacoes['review_text_combined']))

# Aplicar o selection sort
sorted_sentiment_reviews_leia = selection_sort_by_value_descending(data_for_sorting_leia)

print("Comentários ordenados pela pontuação de sentimento (mais positiva para mais negativa - LeIA):")
for score, review in sorted_sentiment_reviews_leia[:10]: # Mostrar os 10 mais positivos
    print(f"({score:.2f}) {review}")

print("\n" + "="*70 + "\n")

print("Comentários ordenados pela pontuação de sentimento (mais negativa para mais positiva - LeIA):")
for score, review in sorted_sentiment_reviews_leia[-10:]: # Mostrar os 10 mais negativos
    print(f"({score:.2f}) {review}")

### **Análise de Discrepâncias Opostas: `review_score` vs. `score_classificado_transformers`**

Vamos identificar e analisar casos em que a classificação do usuário (`review_score`) e a classificação do modelo (`score_classificado_transformers`) são extremas e opostas (e.g., usuário deu 1 estrela, mas o modelo classificou como 5 estrelas, ou vice-versa). Isso pode indicar erros de interpretação do modelo ou talvez até sarcasmo/ironia no texto que o modelo não conseguiu capturar.

In [ ]:
total_avaliacoes = len(df_avaliacoes)

# Calcular acertos para VADER
acertos_vader = (df_avaliacoes['review_score'] == df_avaliacoes['score_classificado_vader']).sum()
percentual_acertos_vader = (acertos_vader / total_avaliacoes) * 100

# Calcular acertos para LeIA
acertos_leia = (df_avaliacoes['review_score'] == df_avaliacoes['score_classificado_leia']).sum()
percentual_acertos_leia = (acertos_leia / total_avaliacoes) * 100

# Calcular acertos para Transformers
acertos_transformers = (df_avaliacoes['review_score'] == df_avaliacoes['score_classificado_transformers']).sum()
percentual_acertos_transformers = (acertos_transformers / total_avaliacoes) * 100

# Criar um DataFrame para exibir os resultados
df_acertos = pd.DataFrame({
    'Modelo': ['VADER', 'LeIA', 'Transformers'],
    'Acertos': [acertos_vader, acertos_leia, acertos_transformers],
    'Total Avaliações': [total_avaliacoes, total_avaliacoes, total_avaliacoes],
    'Percentual de Acertos': [f'{percentual_acertos_vader:.2f}%', f'{percentual_acertos_leia:.2f}%', f'{percentual_acertos_transformers:.2f}%']
})

print("\n=== Percentual de Acertos dos Modelos de Sentimento vs. Review Score Original ===")
display(df_acertos)

## **CASO 1: Avaliações com Nota Baixa (1 Estrela) mas Sentimento Positivo (LeIA)**

Esta seção analisa avaliações onde o cliente deu uma nota muito baixa (1 estrela), mas o modelo LeIA classificou o sentimento como 'Muito Positivo'. Isso pode indicar sarcasmo, ironia, ou uma falha do modelo em compreender o contexto da avaliação.

In [ ]:
colunas_exibir_leia = [
    'review_text_combined', 'review_score',
    'pontuacao_sentimento_leia', 'classe_leia_nova',
]

# CASO 1: NOTA BAIXA (1) COM SENTIMENTO MUITO POSITIVO (LeIA)
caso_1 = df_avaliacoes[
    (df_avaliacoes['review_score'] == 1) &
    (df_avaliacoes['classe_leia_nova'].isin(['muito positivo']))
].copy()

# Ordenar pelos mais positivos segundo o LeIA
caso_1_sorted = caso_1.sort_values(by='pontuacao_sentimento_leia', ascending=False)

print("=" * 120)
print("CASO 1: AVALIAÇÕES BAIXAS (1 ESTRELA) COM SENTIMENTO MUITO POSITIVO (LeIA)")
print("=" * 120)
display(caso_1_sorted[colunas_exibir_leia].head(30))

## **CASO 2: Avaliações com Nota Alta (5 Estrelas) mas Sentimento Negativo (LeIA)**

Esta seção explora avaliações onde o cliente atribuiu uma nota muito alta (5 estrelas), mas o modelo LeIA detectou um sentimento 'Muito Negativo'. Isso também pode apontar para nuances textuais complexas, como uma reclamação detalhada seguida por uma avaliação alta da resolução do problema, ou outras dificuldades de interpretação do modelo.

In [ ]:
# CASO 2: NOTA ALTA (5) COM SENTIMENTO MUITO NEGATIVO (LeIA)
caso_2 = df_avaliacoes[
    (df_avaliacoes['review_score'] == 5) &
    (df_avaliacoes['classe_leia_nova'].isin(['muito negativo']))
].copy()

# Ordenar pelos mais negativos segundo o LeIA
caso_2_sorted = caso_2.sort_values(by='pontuacao_sentimento_leia', ascending=True)

print("\n" + "=" * 120)
print("CASO 2: AVALIAÇÕES ALTAS (5 ESTRELAS) COM SENTIMENTO MUITO NEGATIVO (LeIA)")
print("=" * 120)
display(caso_2_sorted[colunas_exibir_leia].head(30))

## **CASO 1: Avaliações com Nota Baixa (1 Estrela) mas Sentimento Positivo (VADER)**

Esta seção detalha avaliações com `review_score` de 1 estrela, mas classificadas como 'Muito Positivo' pelo VADER, explorando possíveis razões para essa contradição.

In [ ]:
colunas_exibir_vader = [
    'review_text_combined', 'review_score',
    'pontuacao_sentimento_vader', 'classe_vader',
]

# CASO 1: NOTA BAIXA (1) COM SENTIMENTO MUITO POSITIVO (VADER)
caso_1_vader = df_avaliacoes[
    (df_avaliacoes['review_score'] == 1) &
    (df_avaliacoes['classe_vader'].isin(['muito positivo']))
].copy()

# Ordenar pelos mais positivos segundo o VADER
caso_1_vader_sorted = caso_1_vader.sort_values(by='pontuacao_sentimento_vader', ascending=False)

print("=" * 120)
print("CASO 1: AVALIAÇÕES BAIXAS (1 ESTRELA) COM SENTIMENTO MUITO POSITIVO (VADER)")
print("=" * 120)
display(caso_1_vader_sorted[colunas_exibir_vader].head(30))

## **CASO 2: Avaliações com Nota Alta (5 Estrelas) mas Sentimento Negativo (VADER)**

Aqui, analisamos avaliações com `review_score` de 5 estrelas que o VADER classificou como 'Muito Negativo', buscando entender as complexidades por trás dessas divergências.

In [ ]:
# CASO 2: NOTA ALTA (5) COM SENTIMENTO MUITO NEGATIVO (VADER)
caso_2_vader = df_avaliacoes[
    (df_avaliacoes['review_score'] == 5) &
    (df_avaliacoes['classe_vader'].isin(['muito negativo']))
].copy()

# Ordenar pelos mais negativos segundo o VADER
caso_2_vader_sorted = caso_2_vader.sort_values(by='pontuacao_sentimento_vader', ascending=True)

print("\n" + "=" * 120)
print("CASO 2: AVALIAÇÕES ALTAS (5 ESTRELAS) COM SENTIMENTO MUITO NEGATIVO (VADER)")
print("=" * 120)
display(caso_2_vader_sorted[colunas_exibir_vader].head(30))

## **CASO 1: Avaliações com Nota Baixa (1 Estrela) mas Sentimento Positivo (Transformers)**

Esta seção apresenta avaliações com `review_score` de 1 estrela, onde o modelo `Transformers` inesperadamente atribuiu um sentimento 'Muito Positivo'.

In [ ]:
colunas_exibir_transformers = [
    'review_text_combined', 'review_score',
    'pontuacao_sentimento_transformers', 'classe_transformers',
]

# CASO 1: NOTA BAIXA (1) COM SENTIMENTO MUITO POSITIVO (Transformers)
caso_1_transformers = df_avaliacoes[
    (df_avaliacoes['review_score'] == 1) &
    (df_avaliacoes['classe_transformers'].isin(['muito positivo']))
].copy()

# Ordenar pelos mais positivos segundo o Transformers
caso_1_transformers_sorted = caso_1_transformers.sort_values(by='pontuacao_sentimento_transformers', ascending=False)

print("=" * 120)
print("CASO 1: AVALIAÇÕES BAIXAS (1 ESTRELA) COM SENTIMENTO MUITO POSITIVO (Transformers)")
print("=" * 120)
display(caso_1_transformers_sorted[colunas_exibir_transformers].head(30))

## **CASO 2: Avaliações com Nota Alta (5 Estrelas) mas Sentimento Negativo (Transformers)**

Aqui, examinamos avaliações com `review_score` de 5 estrelas que o modelo `Transformers` classificou como 'Muito Negativo', investigando as nuances de sua interpretação.

In [ ]:
# CASO 2: NOTA ALTA (5) COM SENTIMENTO MUITO NEGATIVO (Transformers)
caso_2_transformers = df_avaliacoes[
    (df_avaliacoes['review_score'] == 5) &
    (df_avaliacoes['classe_transformers'].isin(['muito negativo']))
].copy()

# Ordenar pelos mais negativos segundo o Transformers
caso_2_transformers_sorted = caso_2_transformers.sort_values(by='pontuacao_sentimento_transformers', ascending=True)

print("\n" + "=" * 120)
print("CASO 2: AVALIAÇÕES ALTAS (5 ESTRELAS) COM SENTIMENTO MUITO NEGATIVO (Transformers)")
print("=" * 120)
display(caso_2_transformers_sorted[colunas_exibir_transformers].head(30))

## **5. Conclusão e Observações Finais**

Esta seção resume os principais insights obtidos da análise de sentimento utilizando os modelos VADER, LeIA e Transformers, destacando suas características e como se comportaram em relação às `review_score`s originais.

### **Performance Geral de Acertos (Concordância com `review_score` original):**

*   **Transformers:** 59.33%
*   **LeIA:** 32.67%
*   **VADER:** 10.05%

### **Análise por Modelo:**

### **VADER:**
*   **Força:** Extremamente rápido e eficiente para grandes volumes de dados. Tende a ser mais conservador, com uma concentração maior de pontuações neutras ou próximas ao zero.
*   **Limitação:** Por ser baseado em léxicos, sua capacidade de capturar nuances, sarcasmo e o contexto complexo do português é limitada. Isso se reflete no menor percentual de acertos e nos casos de contradição ('Caso 1' e 'Caso 2'), onde o modelo teve dificuldade em alinhar-se com avaliações extremas (1 ou 5 estrelas) que continham palavras-chave que induziram a uma interpretação oposta.

### **LeIA:**
*   **Força:** Desenvolvido especificamente para o português, o LeIA apresenta uma vantagem significativa na compreensão de idiomatismos e estruturas gramaticais locais em comparação com o VADER. Seu percentual de acertos é consideravelmente superior ao VADER, mostrando um melhor equilíbrio entre velocidade e acurácia para o idioma.
*   **Limitação:** Embora melhor que o VADER para o português, por ainda ser um modelo léxico-baseado, o LeIA pode não capturar completamente o contexto de frases mais complexas ou ambíguas, especialmente em casos de reviews muito curtos ou com elementos que podem ser interpretados de múltiplas formas (vide 'Caso 1' e 'Caso 2').

### **Transformers (nlptown/bert-base-multilingual-uncased-sentiment):**
*   **Força:** Com base em aprendizado de máquina pré-treinado, o modelo Transformers demonstra um entendimento contextual profundo do texto. Isso o torna o mais preciso dos três para esta tarefa, com quase 60% de acertos, sendo capaz de capturar sentimentos com maior precisão mesmo em frases complexas e lidando bem com a complexidade do idioma português. Os exemplos nos 'Caso 1' e 'Caso 2' mostram que, embora ainda existam contradições, o modelo lida melhor com a interpretação contextual.
*   **Limitação:** É o mais lento dos três devido à sua complexidade e demanda computacional. O mapeamento de 'estrelas' para uma pontuação contínua, embora necessário para a comparação, pode introduzir alguma granularidade artificial.

### **Observações Gerais e Casos de Contradição:**
*   **Discrepâncias:** A análise detalhada dos 'Caso 1' (review de 1 estrela classificada como muito positiva) e 'Caso 2' (review de 5 estrelas classificada como muito negativa) para todos os modelos revelou que as contradições são um desafio inerente à análise de sentimento. O VADER apresentou o menor número de contradições extremas, mas também o menor percentual de acertos geral. O LeIA mostrou um bom desempenho intermediário. O Transformers, apesar de ter o maior número de contradições em termos absolutos (devido à sua sensibilidade e complexidade), também obteve o maior percentual de acertos, indicando que suas 'falhas' são mais sofisticadas e contextuais.
*   **Complexidade do Idioma:** O português, com seu uso rico de sarcasmo, ironia e nuances culturais, representa um desafio considerável para qualquer modelo de análise de sentimento. Modelos baseados em redes neurais (como Transformers) tendem a ser mais eficazes na compreensão dessas complexidades.
*   **Escolha do Modelo:** A seleção do modelo ideal deve ponderar entre **velocidade** e **precisão**. Para análises em larga escala onde a velocidade é crítica, VADER ou LeIA podem ser opções razoáveis, especialmente se a acurácia contextual for secundária. No entanto, para uma compreensão profunda e precisa do sentimento do cliente, especialmente em cenários críticos onde a nuance é importante, o modelo Transformers é a escolha superior, justificando o maior custo computacional.